# HTTP Request Risk Scoring


In [39]:
from pathlib import Path
import shutil
import urllib.request

try:
    from google.colab import drive
    IS_COLAB = True
except ModuleNotFoundError:
    drive = None
    IS_COLAB = False

DRIVE_ROOT = None
DRIVE_PROCESSED_DIR = None

LOCAL_PROJECT_ROOT = next((
    path for path in (Path.cwd(), Path.cwd().parent, Path('/content/HTTP-Request-Risk-Scoring-Decision-Tree'))
    if (path / 'src' / 'feature_extractor.py').exists()
), None)

if IS_COLAB:
    drive.mount('/content/drive')

    DRIVE_ROOT = next((path for path in (Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')) if path.exists()), None)
    if DRIVE_ROOT is None:
        raise FileNotFoundError('Google Drive mounted, but MyDrive was not found under /content/drive')

    DRIVE_PROCESSED_DIR = DRIVE_ROOT / 'processed'
    DRIVE_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    if LOCAL_PROJECT_ROOT is not None:
        local_features = LOCAL_PROJECT_ROOT / 'data' / 'processed' / 'features.csv'
        if local_features.exists() and not (DRIVE_PROCESSED_DIR / 'features.csv').exists():
            shutil.copy2(local_features, DRIVE_PROCESSED_DIR / 'features.csv')

        local_src = LOCAL_PROJECT_ROOT / 'src'
        drive_src = DRIVE_PROCESSED_DIR / 'src'
        if local_src.exists():
            shutil.copytree(local_src, drive_src, dirs_exist_ok=True)

    GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/nglong05/HTTP-Request-Risk-Scoring-Decision-Tree/main'

    def _download_if_missing(relative_path, destination):
        if destination.exists():
            return
        destination.parent.mkdir(parents=True, exist_ok=True)
        url = f'{GITHUB_RAW_BASE}/{relative_path}'
        try:
            urllib.request.urlretrieve(url, destination)
            print(f'Downloaded {relative_path}')
        except Exception as exc:
            if destination.exists():
                destination.unlink()
            print(f'Could not download {relative_path}: {exc}')

    _download_if_missing('data/processed/features.csv', DRIVE_PROCESSED_DIR / 'features.csv')
    _download_if_missing('data/raw/samplectf.xml', DRIVE_PROCESSED_DIR / 'samplectf.xml')
    for filename in ['feature_extractor.py', 'request_canonicalizer.py', 'burp_xml_converter.py']:
        _download_if_missing(f'src/{filename}', DRIVE_PROCESSED_DIR / 'src' / filename)
    (DRIVE_PROCESSED_DIR / 'src' / '__init__.py').touch(exist_ok=True)

    missing = []
    if not (DRIVE_PROCESSED_DIR / 'features.csv').exists():
        missing.append('features.csv')
    if not (DRIVE_PROCESSED_DIR / 'src' / 'feature_extractor.py').exists():
        missing.append('src/feature_extractor.py')

    print(f'Drive root: {DRIVE_ROOT}')
    print(f'Processed folder: {DRIVE_PROCESSED_DIR}')
    if missing:
        print('Missing required files: ' + ', '.join(missing))
        print('Upload features.csv and the src folder into the processed folder before running the next cell.')
    else:
        print('\n'.join(sorted(path.name for path in DRIVE_PROCESSED_DIR.iterdir())))
else:
    print('google.colab is not available; skipping Google Drive setup.')
    if LOCAL_PROJECT_ROOT is None:
        print('Local project root was not found. Run this notebook from the project root or its notebooks folder.')
    else:
        print(f'Local project root: {LOCAL_PROJECT_ROOT}')
        print(f'Processed folder: {LOCAL_PROJECT_ROOT / "data" / "processed"}')


google.colab is not available; skipping Google Drive setup.
Local project root: c:\Users\toanv\OneDrive\Desktop\BTL_AI\HTTP-Request-Risk-Scoring-Decision-Tree
Processed folder: c:\Users\toanv\OneDrive\Desktop\BTL_AI\HTTP-Request-Risk-Scoring-Decision-Tree\data\processed


In [40]:
from pathlib import Path
import json
import importlib
import importlib.util
import sys
import types
import joblib
import pandas as pd
from urllib.parse import urlsplit
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score

COLAB_ROOTS = []
if 'DRIVE_PROCESSED_DIR' in globals() and DRIVE_PROCESSED_DIR is not None:
    COLAB_ROOTS.append(DRIVE_PROCESSED_DIR)
for root in (Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')):
    if root.exists():
        COLAB_ROOTS.extend([
            root / 'processed',
            root / 'BTL_AI' / 'HTTP-Request-Risk-Scoring-Decision-Tree',
            root / 'HTTP-Request-Risk-Scoring-Decision-Tree',
        ])

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent, *COLAB_ROOTS]

CODE_ROOT = next((path for path in ROOT_CANDIDATES if (path / 'src' / 'feature_extractor.py').exists()), None)
if CODE_ROOT is None:
    checked = '\n'.join(f'- {path}' for path in ROOT_CANDIDATES)
    upload_hint = DRIVE_PROCESSED_DIR / 'src' if 'DRIVE_PROCESSED_DIR' in globals() else 'MyDrive/processed/src'
    raise FileNotFoundError(
        f'Cannot find src/feature_extractor.py. Upload the src folder to {upload_hint}.\n'
        f'Checked:\n{checked}'
    )

DATA_CANDIDATES = []
if 'DRIVE_PROCESSED_DIR' in globals() and DRIVE_PROCESSED_DIR is not None:
    DATA_CANDIDATES.append(DRIVE_PROCESSED_DIR)
DATA_CANDIDATES.extend([CODE_ROOT / 'data' / 'processed', CODE_ROOT, *COLAB_ROOTS])
PROJECT_ROOT = next((path for path in DATA_CANDIDATES if (path / 'features.csv').exists()), None)
if PROJECT_ROOT is None:
    checked = '\n'.join(f'- {path}' for path in DATA_CANDIDATES)
    upload_hint = DRIVE_PROCESSED_DIR / 'features.csv' if 'DRIVE_PROCESSED_DIR' in globals() else 'MyDrive/processed/features.csv'
    raise FileNotFoundError(
        f'Cannot find features.csv. Upload it to {upload_hint}.\n'
        f'Checked:\n{checked}'
    )

SRC_ROOT = CODE_ROOT / 'src'
if not SRC_ROOT.exists() and CODE_ROOT.name == 'src':
    SRC_ROOT = CODE_ROOT
    CODE_ROOT = CODE_ROOT.parent

(SRC_ROOT / '__init__.py').touch(exist_ok=True)
for path in (CODE_ROOT, SRC_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
importlib.invalidate_caches()

try:
    from src.feature_extractor import extract_features
    from src.burp_xml_converter import convert_burp_xml_to_csv
except ModuleNotFoundError:
    src_package = types.ModuleType('src')
    src_package.__path__ = [str(SRC_ROOT)]
    sys.modules['src'] = src_package
    for module_name in ('request_canonicalizer', 'burp_xml_converter', 'feature_extractor'):
        module_path = SRC_ROOT / f'{module_name}.py'
        spec = importlib.util.spec_from_file_location(f'src.{module_name}', module_path)
        module = importlib.util.module_from_spec(spec)
        sys.modules[f'src.{module_name}'] = module
        spec.loader.exec_module(module)
    extract_features = sys.modules['src.feature_extractor'].extract_features
    convert_burp_xml_to_csv = sys.modules['src.burp_xml_converter'].convert_burp_xml_to_csv

print(f'Code root: {CODE_ROOT}')
print(f'Data root: {PROJECT_ROOT}')
print('imports ok')



Code root: c:\Users\toanv\OneDrive\Desktop\BTL_AI\HTTP-Request-Risk-Scoring-Decision-Tree
Data root: c:\Users\toanv\OneDrive\Desktop\BTL_AI\HTTP-Request-Risk-Scoring-Decision-Tree\data\processed
imports ok


In [41]:
features_path = PROJECT_ROOT / 'features.csv'
df = pd.read_csv(features_path)

display(df.shape)
display(df['label'].value_counts().sort_index().to_frame('count'))
display(df['source'].value_counts().to_frame('count'))


(10332, 72)

,count
label,
0,6810
1,3522


,count
source,
openappsec_legitimate,3000
modsec_learn_legitimate,2990
csic_2010,1483
openappsec_malicious,1053
modsec_learn_sqli,789
exploitdb,513
owasp_modsecurity,504


In [42]:
meta_cols = {'id', 'label', 'source', 'notes'}
feature_cols = [col for col in df.columns if col not in meta_cols]

X = df[feature_cols]
y = df['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

THRESHOLD = 5.0
MODEL_CONFIGS = {
    'decision_tree': DecisionTreeClassifier(max_depth=8, min_samples_leaf=20, random_state=42),
    'random_forest': RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=42,
    ),
}

fitted_models = {}
evaluation_rows = []
model_predictions = {}

for model_name, estimator in MODEL_CONFIGS.items():
    estimator.fit(X_train, y_train)
    scores = estimator.predict_proba(X_test)[:, 1] * 10
    preds = (scores >= THRESHOLD).astype(int)

    fitted_models[model_name] = estimator
    model_predictions[model_name] = {'risk_score': scores, 'pred_label': preds}
    evaluation_rows.append({
        'model': model_name,
        'accuracy_pct': round(accuracy_score(y_test, preds) * 100, 2),
        'roc_auc': round(roc_auc_score(y_test, scores / 10), 4),
        'avg_precision': round(average_precision_score(y_test, scores / 10), 4),
        'pred_normal': int((preds == 0).sum()),
        'pred_attack': int((preds == 1).sum()),
    })

    print(f'{model_name} accuracy: {accuracy_score(y_test, preds) * 100:.2f}%')
    display(pd.DataFrame(
        confusion_matrix(y_test, preds, labels=[0, 1]),
        index=['true_normal', 'true_attack'],
        columns=['pred_normal', 'pred_attack'],
    ))

comparison = pd.DataFrame(evaluation_rows).sort_values(
    ['roc_auc', 'avg_precision', 'accuracy_pct'],
    ascending=False,
)
display(comparison)

ACTIVE_MODEL_NAME = 'random_forest'
model = fitted_models[ACTIVE_MODEL_NAME]
y_prob = model.predict_proba(X_test)[:, 1]
risk_score = y_prob * 10
y_pred = (risk_score >= THRESHOLD).astype(int)

print(f'Active model: {ACTIVE_MODEL_NAME}')
print(f'Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%')

display(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[0, 1]),
    index=['true_normal', 'true_attack'],
    columns=['pred_normal', 'pred_attack'],
))

Path('models').mkdir(exist_ok=True)
# joblib.dump(
#     {'model': model, 'model_name': ACTIVE_MODEL_NAME, 'feature_columns': feature_cols, 'threshold': THRESHOLD},
#     f'models/{ACTIVE_MODEL_NAME}.joblib',
# )


decision_tree accuracy: 94.53%


,pred_normal,pred_attack
true_normal,1337,25
true_attack,88,617


random_forest accuracy: 94.10%


,pred_normal,pred_attack
true_normal,1291,71
true_attack,51,654


,model,accuracy_pct,roc_auc,avg_precision,pred_normal,pred_attack
1,random_forest,94.10,0.9897,0.9818,1342,725
0,decision_tree,94.53,0.9836,0.9736,1425,642


Active model: random_forest
Accuracy: 94.10%


,pred_normal,pred_attack
true_normal,1291,71
true_attack,51,654


In [43]:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)
print(f'Feature importance for active model: {ACTIVE_MODEL_NAME}')
display(importance.head(15))


Feature importance for active model: random_forest


,feature,importance
7,num_params,0.151236
25,has_numeric_value,0.108040
34,raw_encoded_ratio,0.098616
49,has_referer_header,0.081989
41,has_recursive_decoded_url,0.074553
27,has_url_value,0.059252
5,path_depth,0.052972
33,decoded_changes_value,0.042568
38,has_recursive_decoded_sqli,0.039899
29,has_base64_like_value,0.033119


In [44]:
test_result = df.loc[X_test.index, ['id', 'source', 'notes', 'label']].copy()
test_result['risk_score'] = risk_score.round(2)
test_result['pred_label'] = y_pred

def source_benchmark(source_name, title):
    data = test_result[test_result['source'] == source_name].copy()
    print(f'{title} test rows: {len(data)}')
    if len(data) == 0:
        return data
    acc = accuracy_score(data['label'].astype(int), data['pred_label'])
    print(f'{title} accuracy: {acc * 100:.2f}%')
    display(pd.DataFrame(confusion_matrix(data['label'].astype(int), data['pred_label'], labels=[0, 1]), index=['true_normal', 'true_attack'], columns=['pred_normal', 'pred_attack']))
    display(data.groupby(['notes', 'label', 'pred_label']).size().reset_index(name='count').sort_values('count', ascending=False).head(15))
    return data

test_csic = source_benchmark('csic_2010', 'CSIC')


CSIC test rows: 304
CSIC accuracy: 72.70%


,pred_normal,pred_attack
true_normal,110,55
true_attack,28,111


,notes,label,pred_label,count
3,csic_normal,0,0,110
1,csic_anomalous,1,1,103
4,csic_normal,0,1,55
0,csic_anomalous,1,0,28
7,csic_xss,1,1,3
5,csic_sqli,1,1,2
6,csic_traversal,1,1,2
2,csic_crlf,1,1,1


In [48]:
def _visible_columns(data):
    candidates = ['id', 'label', 'source', 'notes', 'method', 'url', 'path', 'query_string']
    return [col for col in candidates if col in data.columns]

def _selected_model(estimator=None):
    return model if estimator is None else estimator

def _score_feature_dataset(data, estimator=None):
    estimator = _selected_model(estimator)
    X_ext = data.reindex(columns=feature_cols, fill_value=0)
    scores = estimator.predict_proba(X_ext)[:, 1] * 10
    out = data[_visible_columns(data)].copy()
    out['risk_score'] = scores.round(2)
    return out

def _score_raw_request_dataset(data, estimator=None):
    estimator = _selected_model(estimator)
    feature_rows = []
    for item in data.fillna('').to_dict('records'):
        req = {key: str(value) for key, value in item.items()}
        features = extract_features(req)
        feature_rows.append({col: features.get(col, 0) for col in feature_cols})
    X_ext = pd.DataFrame(feature_rows).reindex(columns=feature_cols, fill_value=0)
    scores = estimator.predict_proba(X_ext)[:, 1] * 10
    out = data[_visible_columns(data)].copy()
    out['risk_score'] = scores.round(2)
    return out

def test_csv(filename, threshold=THRESHOLD, top_n=30, estimator=None, model_name=None):
    selected_model = _selected_model(estimator)
    selected_name = model_name or globals().get('ACTIVE_MODEL_NAME', 'custom_model')
    path = PROJECT_ROOT / filename
    data = pd.read_csv(path)
    mode = 'feature' if set(feature_cols).issubset(data.columns) else 'raw_request'
    result = _score_feature_dataset(data, selected_model) if mode == 'feature' else _score_raw_request_dataset(data, selected_model)
    result['pred_label'] = (result['risk_score'] >= threshold).astype(int)
    result['predicted'] = result['pred_label'].map({0: 'normal', 1: 'attack'})

    print(f'file: {path}')
    print(f'model: {selected_name}')
    print(f'mode: {mode}')
    print(f'rows: {len(result)}')
    print(f'threshold: {threshold}')

    if 'label' in result.columns:
        y_true = result['label'].astype(int)
        y_pred_ext = result['pred_label'].astype(int)
        print(f'accuracy: {accuracy_score(y_true, y_pred_ext) * 100:.2f}%')
        print(classification_report(
            y_true,
            y_pred_ext,
            labels=[0, 1],
            target_names=['normal', 'attack'],
            zero_division=0,
            digits=4,
        ))
        display(pd.DataFrame(
            confusion_matrix(y_true, y_pred_ext, labels=[0, 1]),
            index=['true_normal', 'true_attack'],
            columns=['pred_normal', 'pred_attack'],
        ))

        threshold_rows = []
        for value in [3.0, 4.0, 5.0, 6.0, 7.0, 8.0]:
            pred = (result['risk_score'] >= value).astype(int)
            threshold_rows.append({
                'threshold': value,
                'accuracy': round(accuracy_score(y_true, pred) * 100, 2),
                'pred_normal': int((pred == 0).sum()),
                'pred_attack': int((pred == 1).sum()),
            })
        display(pd.DataFrame(threshold_rows))

    display(result.sort_values('risk_score', ascending=False).head(top_n))
    return result


In [49]:
BURP_XML_FILE = 'samplectf.xml'
BURP_LABEL = 0
BURP_OUTPUT_CSV = f'{Path(BURP_XML_FILE).stem}_requests.csv'

def _find_input_file(filename):
    candidates = [
        PROJECT_ROOT / filename,
        Path.cwd() / filename,
    ]
    if 'CODE_ROOT' in globals() and CODE_ROOT is not None:
        candidates.extend([
            CODE_ROOT / filename,
            CODE_ROOT / 'data' / 'raw' / filename,
            CODE_ROOT / 'data' / 'processed' / filename,
        ])
    if 'DRIVE_PROCESSED_DIR' in globals() and DRIVE_PROCESSED_DIR is not None:
        candidates.append(DRIVE_PROCESSED_DIR / filename)

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        candidate = Path(candidate)
        key = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)

    return next((candidate for candidate in unique_candidates if candidate.exists()), None), unique_candidates

burp_xml_path, burp_xml_candidates = _find_input_file(BURP_XML_FILE)
burp_csv_path = PROJECT_ROOT / BURP_OUTPUT_CSV

if burp_xml_path is None:
    print(f'Skipping Burp XML test; file not found: {BURP_XML_FILE}')
    print('Checked:')
    for candidate in burp_xml_candidates:
        print(f'- {candidate}')
    burp_xml_result = None
else:
    count = convert_burp_xml_to_csv(
        burp_xml_path,
        burp_csv_path,
        label=BURP_LABEL,
        source=f'burp_{Path(BURP_XML_FILE).stem}',
        notes='burp_external',
        id_prefix='burp_external',
    )
    print(f'Input XML: {burp_xml_path}')
    print(f'Converted {count} Burp requests to {burp_csv_path}')
    burp_xml_result = test_csv(BURP_OUTPUT_CSV, threshold=THRESHOLD, top_n=30)


Input XML: c:\Users\toanv\OneDrive\Desktop\BTL_AI\HTTP-Request-Risk-Scoring-Decision-Tree\data\raw\samplectf.xml
Converted 6 Burp requests to c:\Users\toanv\OneDrive\Desktop\BTL_AI\HTTP-Request-Risk-Scoring-Decision-Tree\data\processed\samplectf_requests.csv
file: c:\Users\toanv\OneDrive\Desktop\BTL_AI\HTTP-Request-Risk-Scoring-Decision-Tree\data\processed\samplectf_requests.csv
model: random_forest
mode: raw_request
rows: 6
threshold: 5.0
accuracy: 50.00%
              precision    recall  f1-score   support

      normal     1.0000    0.5000    0.6667         6
      attack     0.0000    0.0000    0.0000         0

    accuracy                         0.5000         6
   macro avg     0.5000    0.2500    0.3333         6
weighted avg     1.0000    0.5000    0.6667         6



,pred_normal,pred_attack
true_normal,3,3
true_attack,0,0


,threshold,accuracy,pred_normal,pred_attack
0,3.0,16.67,1,5
1,4.0,50.00,3,3
2,5.0,50.00,3,3
3,6.0,50.00,3,3
4,7.0,50.00,3,3
5,8.0,66.67,4,2


,id,label,source,notes,method,url,path,query_string,risk_score,pred_label,predicted
0,burp_external_000001,0,burp_samplectf,burp_external,GET,http://localhost:3000/somepath?file=%252E%252E...,/somepath,file=%252E%252E%252F%252E%252E%252F%252E%252E%...,9.33,1,attack
5,burp_external_000006,0,burp_samplectf,burp_external,GET,https://www.google.com/warmup.html,/warmup.html,NaN,8.36,1,attack
4,burp_external_000005,0,burp_samplectf,burp_external,GET,http://localhost:3000/,/,NaN,7.18,1,attack
3,burp_external_000004,0,burp_samplectf,burp_external,GET,http://localhost:3000/app.js,/app.js,NaN,3.30,0,normal
2,burp_external_000003,0,burp_samplectf,burp_external,GET,http://localhost:3000/favicon.ico,/favicon.ico,NaN,3.30,0,normal
1,burp_external_000002,0,burp_samplectf,burp_external,GET,http://localhost:3000/api/coolbeans,/api/coolbeans,NaN,2.86,0,normal


In [47]:
TEST_FILE = 'features.csv'
external_result = test_csv(TEST_FILE, threshold=THRESHOLD, top_n=30)

file: c:\Users\toanv\OneDrive\Desktop\BTL_AI\HTTP-Request-Risk-Scoring-Decision-Tree\data\processed\features.csv
model: random_forest
mode: feature
rows: 10332
threshold: 5.0
accuracy: 94.91%
              precision    recall  f1-score   support

      normal     0.9685    0.9537    0.9611      6810
      attack     0.9131    0.9401    0.9264      3522

    accuracy                         0.9491     10332
   macro avg     0.9408    0.9469    0.9437     10332
weighted avg     0.9496    0.9491    0.9493     10332



,pred_normal,pred_attack
true_normal,6495,315
true_attack,211,3311


,threshold,accuracy,pred_normal,pred_attack
0,3.0,89.80,5808,4524
1,4.0,92.83,6273,4059
2,5.0,94.91,6706,3626
3,6.0,95.79,7091,3241
4,7.0,94.13,7315,3017
5,8.0,92.61,7482,2850


,id,label,source,notes,risk_score,pred_label,predicted
5262,pos_modsec_learn_sqli_002371,1,modsec_learn_sqli,modsec_learn_sqli:sqlmap,9.95,1,attack
2202,pos_modsec_learn_sqli_002686,1,modsec_learn_sqli,modsec_learn_sqli:sqlmap,9.95,1,attack
4415,pos_modsec_learn_sqli_002349,1,modsec_learn_sqli,modsec_learn_sqli:sqlmap,9.95,1,attack
4414,pos_modsec_learn_sqli_000804,1,modsec_learn_sqli,modsec_learn_sqli:httpparams,9.95,1,attack
6634,pos_modsec_learn_sqli_000524,1,modsec_learn_sqli,modsec_learn_sqli:httpparams,9.95,1,attack
1429,pos_malicious_001427,1,openappsec_malicious,sqli,9.95,1,attack
1422,pos_malicious_001484,1,openappsec_malicious,sqli,9.95,1,attack
8478,pos_modsec_learn_sqli_002652,1,modsec_learn_sqli,modsec_learn_sqli:sqlmap,9.95,1,attack
484,pos_modsec_learn_sqli_002724,1,modsec_learn_sqli,modsec_learn_sqli:sqlmap,9.95,1,attack
6585,pos_modsec_learn_sqli_001617,1,modsec_learn_sqli,modsec_learn_sqli:sqli_kaggle,9.95,1,attack
